In [1]:
# ===== CELL 1 — INSTALLS (datasets pinned: script datasets break on 3.x) =====
import subprocess, sys
for p in ["transformers>=4.44","sentencepiece","accelerate>=0.30","bitsandbytes","datasets==2.19.0","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",p],check=False)
print("ok")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


ok


In [2]:
# ===== CELL 2 — CONFIG · SEEDS · SECRETS · METRIC SWITCH =====
import os, re, gc, glob, json, random, unicodedata, warnings, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
warnings.filterwarnings("ignore"); T0=time.time()
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("harryPotter")
    except Exception: return None
HF_TOKEN=get_hf_token()

@dataclass
class CFG:
    comp_dir:str="/kaggle/input/competitions/bengali-hallucination"
    nli_tsv:str="/kaggle/input/datasets/ajmainmahtab/bangla-natural-language-inference-dataset/NLI Dataset - Combined.tsv"
    wiki_dir:str="/kaggle/input/datasets/disisbig/bengali-wikipedia-articles"
    hf_squad:str="csebuetnlp/squad_bn"
    hf_tydi:tuple=("tydiqa","google-research-datasets/tydiqa")
    hf_ixnli:str="Divyanshu/indicxnli"
    hf_banglarqa:str="sartajekram/BanglaRQA"
    hf_indicqa:tuple=("ai4bharat/IndicQA","indicqa.bn")
    hf_qa_70k: str = "rasheduzzaman/Bangla_question_answer_pair_70K_dataset"
    bhe_qa_1000 = "/kaggle/input/datasets/zoayriaabedin/benhallueval/banglahallueval_qa_1000.csv"
    bhe_qa_dataset = "/kaggle/input/datasets/zoayriaabedin/benhallueval/banglahallueval_qa_dataset.csv"
    bhe_qa_dataset_1000 = "/kaggle/input/datasets/zoayriaabedin/benhallueval/banglahallueval_qa_dataset_1000.csv"
    extra_banks:tuple=()          # HF banks: (repo_id, config_or_None, q_field, a_field)
    extra_bank_files:tuple=()     # Kaggle-dataset banks: (csv_or_json_path, q_col, a_col)
    metric:str="macro"             # Kaggle Overview: "evaluated on macro-F1" (platform wins for LB). Rulebook says f1_c0 — flip back for Phase-2 if organizers confirm.
    backbones:tuple=(("bbl","csebuetnlp/banglabert_large"),)   # user directive: banglabert-large ONLY
    n_seeds_bbl:int=2                                          # seed-bagging recovers ensemble variance
    # mDeBERTa removed after LB ablation (0.681->0.689 without it) + val AUC dominance of bb.
    judge_chain:tuple=("Qwen/Qwen2.5-32B-Instruct",)
    judge_prefer_32b:bool=False   # 32B: last-resort fallback by default; True = try it first (see header budget note)
    max_len:int=256; infer_max_len:int=384; batch_size:int=16; warmup:float=0.10
    backbone_cfg:dict=None    # set post-init: per-model lr/epochs (log-driven)
    focal_gamma:float=2.0; focal_alpha:float=1.0   # no class bias under macro
    n_wiki_files:int=8000; max_train_rows:int=70000   # budget-derived (see header)
    use_retrieval:bool=True; n_passages:int=250000; retr_topk:int=3
    use_llm_suite:bool=True; use_math_solver:bool=True    # ON: 179/180 math rows are no_ctx; val-gated
    enable_overrides:bool=True    # tiered, val-gated
    n_boot:int=200
    judge_ids:tuple=("Qwen/Qwen2.5-32B-Instruct",)
    
cfg=CFG()
cfg.backbone_cfg={"bbl":{"lr":1e-5,"epochs":3}}   # evidence: 8e-5 collapsed; 8e-6 undertrains; guard halves if needed

# ---------- REPORT infrastructure ----------
REPORT_DIR="/kaggle/working/report"; os.makedirs(REPORT_DIR,exist_ok=True)
REPORT={"timings":{}}
def rep(key,obj):
    REPORT[key]=obj
    try:
        with open(os.path.join(REPORT_DIR,f"{key}.json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=str)
    except Exception as e: print("rep skip",key,str(e)[:60])
def rep_df(key,df):
    try: df.to_csv(os.path.join(REPORT_DIR,f"{key}.csv"),index=False)
    except Exception as e: print("rep_df skip",key,str(e)[:60])
import dataclasses
rep("config",dataclasses.asdict(cfg))
def tleft(): print(f"[t+{(time.time()-T0)/60:.1f}m]")
print("device:",DEVICE,"| gpus:",torch.cuda.device_count(),"| metric:",cfg.metric,"| HF token:",("set" if HF_TOKEN else "None (ok)"))

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True

seed_everything(SEED)

device: cuda | gpus: 2 | metric: macro | HF token: set


In [3]:
# ===== CELL 3 — UTILS (+ word-numbers, fuzzy, math detector v2) =====
BN_DIGITS="০১২৩৪৫৬৭৮৯"; BN2ASCII={ord(b):str(i) for i,b in enumerate(BN_DIGITS)}
NULLS={"[null]","null","none","nan","n/a",""}; _P=set("।,.?!;:\"'()[]{}<>/\\|–—’‘“”…%°")
BN_CHAR=re.compile(r"[\u0980-\u09FF]"); LAT=re.compile(r"[A-Za-z]"); DIGIT_RE=re.compile(r"[০-৯0-9]+")
def norm(s): return unicodedata.normalize("NFC",str(s))
def denum(s): return norm(s).translate(BN2ASCII)
def is_no_ctx(c):
    if c is None or (isinstance(c,float) and pd.isna(c)): return True
    return str(c).strip().lower() in NULLS
def toks(s): return [t for t in "".join(" " if c in _P else c for c in denum(s)).split() if t]
def numset(s): return set(re.findall(r"\d+(?:\.\d+)?",denum(s)))
def content(s,m=2): return {t for t in toks(s) if len(t)>=m}
def contain(resp,src):
    r,s=content(resp),content(src); return len(r&s)/len(r) if r else 1.0
def sent_split(t):
    parts=re.split(r"(?<=[।!?])\s+|\n+",norm(t))
    return [p.strip() for p in parts if len(p.strip())>8]
def mostly_bengali(s):
    s=str(s); b=len(BN_CHAR.findall(s)); return b>=max(1,int(0.5*len(re.findall(r"\S",s))))
def latin_frac(s):
    s=str(s); b=len(BN_CHAR.findall(s)); l=len(LAT.findall(s)); return l/max(1,b+l)
def bump_digits(a):
    def b(ch):
        if ch in BN_DIGITS: 
            return BN_DIGITS[(BN_DIGITS.index(ch)+random.randint(1,8))%10]
        # Changed isdigit() to isdecimal() to avoid crashing on superscripts like '²'
        if ch.isdecimal(): 
            return str((int(ch)+random.randint(1,8))%10)
        return ch
    n="".join(b(c) for c in a)
    return None if n==a else n
def normr(s):  # aggressive response normalizer for matching
    s=denum(s); s=re.sub(r"[।,.?!;:'\"()\[\]{}%-]"," ",s)
    return re.sub(r"\s+"," ",s).strip().lower()
WORDNUM={"শূন্য":0,"এক":1,"দুই":2,"তিন":3,"চার":4,"পাঁচ":5,"ছয়":6,"সাত":7,"আট":8,"নয়":9,"দশ":10,
 "এগারো":11,"বারো":12,"পনেরো":15,"বিশ":20,"পঁচিশ":25,"ত্রিশ":30,"চল্লিশ":40,"পঞ্চাশ":50,
 "ষাট":60,"সত্তর":70,"আশি":80,"নব্বই":90,"একশ":100,"শত":100,"হাজার":1000,"লাখ":100000,"লক্ষ":100000,"কোটি":10000000}
def word2num(s):
    s=normr(s); total=0; cur=0; seen=False
    for w in s.split():
        if w in WORDNUM:
            v=WORDNUM[w]; seen=True
            if v>=100: cur=max(cur,1)*v; total+=cur; cur=0
            else: cur+=v
        elif re.fullmatch(r"\d+(?:\.\d+)?",w): cur+=float(w); seen=True
    total+=cur
    return total if seen else None
def extract_nums(s):
    out=[float(x) for x in re.findall(r"\d+(?:\.\d+)?",denum(s))]
    w=word2num(s)
    if w is not None and w not in out: out.append(float(w))
    return out
def ngr(s,n=3):
    s=normr(s); return {s[i:i+n] for i in range(max(1,len(s)-n+1))}
def fuzzy(a,b):
    A,B=ngr(a),ngr(b); return len(A&B)/max(1,len(A|B))
MATH_KW=re.compile(r"যোগফল|বিয়োগফল|গুণফল|ভাগফল|শতকরা|সমীকরণ|গড়|অনুপাত|ঘণ্টায়|কিমি|মিনিটে|টাকায়|লাভ|ক্ষতি|মিশ্রণ|কাজটি|বেগে|সুদ|আসল")
def is_math(p):
    p=str(p)
    return bool(DIGIT_RE.search(p)) and (bool(MATH_KW.search(p)) or bool(re.search(r"[+×*/=^]|(?<=\d)\s*-\s*(?=\d)",denum(p))))

In [4]:
# ===== CELL 4 — COMPETITION DATA & 70K DATASET MERGE =====
from datasets import load_dataset

# 1. Load standard competition data
sample=pd.DataFrame(json.load(open(os.path.join(cfg.comp_dir,"dataset samples.json"),encoding="utf-8")))
test=pd.read_csv(os.path.join(cfg.comp_dir,"test set.csv"))
sub=pd.read_csv(os.path.join(cfg.comp_dir,"sample submission.csv"))

for df in (sample,test):
    df["no_ctx"]=df["context"].map(is_no_ctx)
    df["ctx_clean"]=df.apply(lambda r:"" if r["no_ctx"] else str(r["context"]),axis=1)
    df["premise"]=df.apply(lambda r: str(r["prompt_bn"]) if r["no_ctx"]
                           else (str(r["prompt_bn"])+" "+r["ctx_clean"]).strip(),axis=1)
    df["response"]=df["response_bn"].astype(str)
    df["mathq"]=df["prompt_bn"].map(is_math)
    VOCRE=re.compile(r"অর্থ|ভাবার্থ|সমার্থক|বিপরীত|বাগধারা|প্রতিশব্দ")
    GRAMRE=re.compile(r"সমাস|সন্ধি|কারক|বিভক্তি|প্রকৃতি|প্রত্যয়|এক কথায়|শুদ্ধ|ব্যাসবাক্য")
    MCQRE=re.compile(r"\(ক\)|\(খ\)|নিচের কোন|কোনটি সঠিক")
    df["vocq"]=df["prompt_bn"].map(lambda s:bool(VOCRE.search(str(s))))
    df["gramq"]=df["prompt_bn"].map(lambda s:bool(GRAMRE.search(str(s))))
    df["mcqq"]=df["prompt_bn"].map(lambda s:bool(MCQRE.search(str(s))))
    df["mixq"]=df.apply(lambda r: latin_frac(str(r["prompt_bn"])+" "+str(r["response_bn"]))>=0.05,axis=1)
assert list(sub.columns)==["id","label"] and len(sub)==len(test)==2516

print("val:",sample.shape,"| test math rows:",int(test.mathq.sum()),
      "| test has/no ctx:",int((~test.no_ctx).sum()),int(test.no_ctx.sum()))

# ---------------------------------------------------------
# 2. NEW: LOAD AND PREPARE THE 70K HUGGINGFACE DATASET
# ---------------------------------------------------------
print("\nDownloading 70k dataset from Hugging Face...")
try:
    hf_dataset = load_dataset(cfg.hf_qa_70k)
    df_70k = hf_dataset['train'].to_pandas()

    # Rename columns to match competition standard
    df_70k = df_70k.rename(columns={
        "question": "prompt_bn",      
        "answer": "response_bn",      
        "context": "ctx_clean"        
    })

    # Add label (assuming all these are correct ground truth = 1)
    if "label" not in df_70k.columns:
        df_70k["label"] = 1  

    # Keep only necessary columns
    cols_to_keep = ["prompt_bn", "response_bn", "label", "ctx_clean"]
    cols_to_keep = [c for c in cols_to_keep if c in df_70k.columns]
    df_70k = df_70k[cols_to_keep]

    # Combine with sample (or your main training DataFrame)
    # If your code uses a different variable for training data later, 
    # you can append df_70k to it there. For now, it's stored in df_70k.
    print(f"Successfully loaded {len(df_70k)} rows from hf_qa_70k!")
    
except Exception as e:
    print(f"Failed to load 70k dataset: {e}")

val: (299, 13) | test math rows: 164 | test has/no ctx: 1361 1155



Generating train split:   0%|          | 0/81072 [00:00<?, ? examples/s]

Successfully loaded 81072 rows from hf_qa_70k!


In [5]:
# ===== CELL 5 — NLI SOURCES (unchanged from v4.1) =====
def load_nli_tsv():
    if not os.path.exists(cfg.nli_tsv): print("tsv missing"); return pd.DataFrame()
    raw=pd.read_csv(cfg.nli_tsv,sep="\t",on_bad_lines="skip").dropna()
    cols={c.lower():c for c in raw.columns}
    def pick(*ns):
        for n in ns:
            for lc,o in cols.items():
                if n in lc: return o
    pc,hc,lc=pick("premise","sentence1","text_a"),pick("hypothesis","sentence2","text_b"),pick("label","gold","class")
    if not(pc and hc and lc):
        # wide format observed in this team's file: [Premise, Entailment, Neutral, Contradiction]
        wide={k:pick(k) for k in ("entailment","neutral","contradiction")}
        pw=pick("premise")
        if pw and all(wide.values()):
            rows=[]
            for _,r in raw.iterrows():
                for k,col in wide.items():
                    h=str(r[col]).strip()
                    if h and h.lower() not in ("nan","none"):
                        rows.append((str(r[pw]),h,1 if k=="entailment" else 0))
            out=pd.DataFrame(rows,columns=["premise","response","label"]); out["src"]="nli"
            print("NLI wide-format melted:",out.shape,out.label.value_counts().to_dict()); return out
        print("cols?",list(raw.columns)); return pd.DataFrame()
    print("--- VERIFY MAPPING (entail must ->1) ---"); print(raw[[pc,hc,lc]].head(3).to_string())
    m={"entailment":1,"entail":1,"neutral":0,"contradiction":0,"contradict":0,"0":1,0:1,"1":0,1:0,"2":0,2:0}
    raw["y"]=raw[lc].astype(str).str.strip().str.lower().map(lambda v:m.get(v,np.nan))
    return pd.DataFrame({"premise":raw[pc].astype(str),"response":raw[hc].astype(str),
                         "label":raw.dropna(subset=["y"])["y"].astype(int),"src":"nli"}).dropna()
def load_indicxnli():
    try:
        from datasets import load_dataset
        d=load_dataset(cfg.hf_ixnli,"bn",split="train",token=HF_TOKEN)
        df=pd.DataFrame({"premise":d["premise"],"response":d["hypothesis"],"label":d["label"]})
        df["label"]=(df["label"]==0).astype(int); df["src"]="ixnli"
        return df.sample(min(40000,len(df)),random_state=SEED)
    except Exception as e:
        print("IndicXNLI skipped:",str(e)[:100]); return pd.DataFrame()
nli_df=pd.concat([load_nli_tsv(),load_indicxnli()],ignore_index=True)
print("NLI total:",nli_df.shape)

NLI wide-format melted: (12600, 4) {0: 8400, 1: 4200}


Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5010 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2490 [00:00<?, ? examples/s]

NLI total: (52600, 4)


In [6]:
# ===== CELL 6 — QA LOADERS (+BanglaRQA, IndicQA) -> triples AND answer banks =====
BANK={}   # normalized question -> set of gold answers (normalized) — fed by every QA source
def bank_add(q,a):
    if q and a: BANK.setdefault(norm(str(q)).strip(),set()).add(normr(a))

def qa_rows(items):
    by_ctx={}
    for it in items: by_ctx.setdefault(it.get("context",""),[]).append(it)
    all_ans=[it["answer"] for it in items if it.get("answer")]
    rows=[]
    for ctx,grp in by_ctx.items():
        answered=[g for g in grp if g.get("answer")]
        
        for g in grp:
            q=str(g["question"]); prem=(q+" "+str(ctx)).strip()
            if g.get("answer"):
                a=str(g["answer"]); rows.append((prem,a,1,"faithful")); bank_add(q,a)
                
                # Optimized negative sampling to prevent O(N^2) notebook freeze
                if len(answered) > 50:
                    candidates = random.sample(answered, 50)
                    others = [str(h["answer"]) for h in candidates if str(h["answer"])!=a]
                else:
                    others = [str(h["answer"]) for h in answered if str(h["answer"])!=a]
                
                if others: rows.append((prem,random.choice(others),0,"wrong_attr"))
                
                if DIGIT_RE.search(a):
                    c=bump_digits(a)
                    if c: rows.append((prem,c,0,"intrinsic"))
                else:
                    for _ in range(6):
                        cand=str(random.choice(all_ans))
                        if cand!=a and cand not in ctx:
                            rows.append((prem,cand,0,"extrinsic")); break
            else:
                w=[t for t in str(ctx).split() if len(t)>2]
                if w:
                    i=random.randrange(max(1,len(w)-2)); rows.append((prem," ".join(w[i:i+2]),0,"unanswerable"))
    return pd.DataFrame(rows,columns=["premise","response","label","mode"])

def _sq_items(d):
    out=[]
    for r in d:
        ans=r["answers"]["text"][0] if r["answers"]["text"] else None
        out.append({"context":r["context"],"question":r["question"],"answer":ans})
    return out

def load_squad_bn():
    out=[]
    from datasets import load_dataset
    for sp in ("train","validation","test"):   
        try: out+=_sq_items(load_dataset(cfg.hf_squad,split=sp,token=HF_TOKEN))
        except Exception as e: print(f"squad_bn[{sp}] skipped:",str(e)[:80])
    return out

def load_tydiqa_bn():
    from datasets import load_dataset
    for rid in cfg.hf_tydi:
        try:
            from datasets import concatenate_datasets
            d=concatenate_datasets([load_dataset(rid,"secondary_task",split=sp,token=HF_TOKEN)
                                    for sp in ("train","validation")])
            return [{"context":r["context"],"question":r["question"],
                     "answer":(r["answers"]["text"][0] if r["answers"]["text"] else None)}
                    for r in d if str(r["id"]).startswith("bengali")]
        except Exception as e: print(f"tydiqa {rid} failed:",str(e)[:80])
    return []

def load_banglarqa():
    try:
        from datasets import load_dataset
        d=load_dataset(cfg.hf_banglarqa,split="train",token=HF_TOKEN)
        out=[]
        for r in d:
            ctx=r.get("context") or r.get("passage") or ""
            q=r.get("question") or r.get("question_text") or ""
            a=r.get("answers") or r.get("answer")
            if isinstance(a,dict): a=(a.get("text") or a.get("answer_text") or [None]); a=a[0] if isinstance(a,list) else a
            if isinstance(a,list): a=a[0] if a else None
            out.append({"context":ctx,"question":q,"answer":(str(a) if a else None)})
        return out
    except Exception as e: print("BanglaRQA skipped:",str(e)[:90]); return []

def load_indicqa():
    try:
        from datasets import load_dataset
        d=load_dataset(cfg.hf_indicqa[0],cfg.hf_indicqa[1],split="test",token=HF_TOKEN,trust_remote_code=True)
        return _sq_items(d)
    except Exception as e: print("IndicQA skipped:",str(e)[:90]); return []

def load_extra_banks():
    from datasets import load_dataset
    n=0
    for spec in cfg.extra_banks:
        rid,cfg_,qf,af=spec
        try:
            d=load_dataset(rid,cfg_,split="train",token=HF_TOKEN) if cfg_ else load_dataset(rid,split="train",token=HF_TOKEN)
            for r in d: bank_add(r.get(qf),r.get(af)); n+=1
        except Exception as e: print("extra bank",rid,"skipped:",str(e)[:80])
    if n: print("extra bank QAs:",n)

PAIRBANK={}   

def load_somadhan_bank():
    try:
        import io
        n=0
        somadhan_path = "/kaggle/input/datasets/zoayriaabedin/somadhan/SOMADHAN.csv"
        df=pd.read_csv(somadhan_path)
        
        cols={c.lower():c for c in df.columns}
        qc=next((cols[k] for k in cols if "question" in k or "problem" in k),df.columns[0])
        ac=next((cols[k] for k in cols if "answer" in k or "solution" in k),df.columns[-1])
        for _,r in df.iterrows():
            sol=str(r[ac])
            tail=sol.split("উত্তর")[-1] if "উত্তর" in sol else sol[-80:]
            nums_=extract_nums(tail)
            if nums_: bank_add(str(r[qc]),str(nums_[-1])); n+=1
        print("SOMADHAN bank entries:",n)
    except Exception as e: print("SOMADHAN skipped:",str(e)[:90])

def _bhe_parse_df(df,tag):
    rows=[]; cols={c.lower().strip():c for c in df.columns}
    def col(*ns):
        for n in ns:
            if n in cols: return cols[n]
    ctxc=col("context","document","ctx","knowledge","passage")
    qc=col("question","prompt","prompt_bn","q")
    ac=col("answer","right answer","right_answer","correct_answer","gold","response","response_bn","a")
    hc=col("hallucinated_answer","hallucinated answer","hallucination","wrong_answer","hallu")
    lc=col("label","target","is_hallucinated")
    for _,r in df.iterrows():
        ctx=str(r[ctxc]) if ctxc and pd.notna(r[ctxc]) else ""
        if ctx.strip().lower() in NULLS: ctx=""
        q=str(r[qc]).strip() if qc and pd.notna(r[qc]) else ""
        if not q: continue
        prem=(q+" "+ctx).strip()
        if ac and hc and pd.notna(r.get(ac)) is not None:
            a=str(r[ac]) if pd.notna(r[ac]) else ""; h=str(r[hc]) if pd.notna(r[hc]) else ""
            if a:
                bank_add(q,a); PAIRBANK[(norm(q).strip(),normr(a))]=1
                rows.append((prem,a,1,"bhe_gold"))
            if h:
                PAIRBANK[(norm(q).strip(),normr(h))]=0
                rows.append((prem,h,0,"bhe_hal"))
        elif ac and lc and pd.notna(r.get(ac)) is not None and pd.notna(r.get(lc)):
            a=str(r[ac]); y=int(float(r[lc]))
            if col("is_hallucinated")==lc: y=1-y     
            PAIRBANK[(norm(q).strip(),normr(a))]=y
            if y==1: bank_add(q,a)
            rows.append((prem,a,y,"bhe_lab"))
        elif ac and pd.notna(r.get(ac)):
            a=str(r[ac]); bank_add(q,a); PAIRBANK[(norm(q).strip(),normr(a))]=1
            rows.append((prem,a,1,"bhe_gold"))
    print(f"  BHE[{tag}]: +{len(rows)} rows")
    return rows

def load_benhallueval():
    rows=[]
    local_paths = [
        "/kaggle/input/datasets/zoayriaabedin/benhallueval/banglahallueval_qa_1000.csv",
        "/kaggle/input/datasets/zoayriaabedin/benhallueval/banglahallueval_qa_dataset.csv",
        "/kaggle/input/datasets/zoayriaabedin/benhallueval/banglahallueval_qa_dataset_1000.csv"
    ]
    
    for p in local_paths:
        if os.path.exists(p):
            try: 
                rows+=_bhe_parse_df(pd.read_csv(p),os.path.basename(p))
            except Exception as e: print("BHE local skip",p,str(e)[:60])
            
    if rows: print("BHE local total:",len(rows))
    print("BenHalluEval: PAIRBANK",len(PAIRBANK),"| train rows",len(rows))
    return rows

def load_bangla_math_hf():
    try:
        from datasets import load_dataset
        d=load_dataset("kawchar85/Bangla-Math",split="train",token=HF_TOKEN)
        cols=d.column_names; n=0
        qc=next((c for c in cols if "question" in c.lower() or "problem" in c.lower()),cols[0])
        ac=next((c for c in cols if "answer" in c.lower() or "solution" in c.lower()),cols[-1])
        for r in d:
            nums_=extract_nums(str(r[ac])[-80:])
            if nums_: bank_add(str(r[qc]),str(nums_[-1])); n+=1
        print("Bangla-Math bank entries:",n)
    except Exception as e: print("Bangla-Math skipped:",str(e)[:80])

import io
def load_qa_70k():
    out=[]; n=0

    try:
        from datasets import load_dataset
        d=load_dataset(cfg.hf_qa_70k, split="train", token=HF_TOKEN)
        
        # DYNAMIC COLUMN FINDER FOR HUGGING FACE: Added 'input' and 'output'
        cols = d.column_names
        cols_lower = {c.lower(): c for c in cols}
        
        q_col = next((cols_lower[k] for k in cols_lower if "question" in k or "prompt" in k or "input" in k), None)
        a_col = next((cols_lower[k] for k in cols_lower if "answer" in k or "response" in k or "output" in k), None)
        
        if not q_col or not a_col:
            print(f"qa_70k (HF): Could not auto-detect question/answer columns in {cols}")
            return []

        for r in d:
            q=str(r.get(q_col,"")).strip(); a=str(r.get(a_col,"")).strip()
            if q and a and q.lower() != "nan" and a.lower() != "nan":
                out.append({"context":"","question":q,"answer":a}); bank_add(q,a); n+=1
        print("qa_70k (HF loaded):", n, "pairs added to BANK")
        return out
    except Exception as e: 
        print("qa_70k skipped:", str(e)[:80])
        return []

# Execute loaders
qa_items=load_squad_bn()+load_tydiqa_bn()+load_banglarqa()+load_indicqa()+load_qa_70k()
load_somadhan_bank(); load_bangla_math_hf()
bhe_rows=load_benhallueval()
random.shuffle(qa_items); qa_items=qa_items[:60000]
load_extra_banks()

# Process rows
qa_df=qa_rows(qa_items) if qa_items else pd.DataFrame(columns=["premise","response","label","mode"])
if len(qa_df): qa_df["src"]="qa"

bhe_df=pd.DataFrame(bhe_rows,columns=["premise","response","label","mode"])
if len(bhe_df): bhe_df["src"]="bhe"; bhe_df=bhe_df.drop_duplicates(subset=["premise","response"])

print("bhe train rows:",len(bhe_df))
print("QA triples:",qa_df.shape,"| BANK size:",len(BANK),"| PAIRBANK:",len(PAIRBANK))
rep("sources",{"qa_items":len(qa_items),"BANK":len(BANK),"PAIRBANK":len(PAIRBANK),"bhe_rows":len(bhe_rows)})
assert len(qa_df)>0, "QA EMPTY — loaders failed; fix before spending GPU hours."

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating train split:   0%|          | 0/49881 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5077 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/1493 [00:00<?, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

qa_70k (HF loaded): 81072 pairs added to BANK
SOMADHAN bank entries: 4001


Generating train split:   0%|          | 0/1455 [00:00<?, ? examples/s]

Bangla-Math bank entries: 1455
  BHE[banglahallueval_qa_1000.csv]: +1000 rows
  BHE[banglahallueval_qa_dataset.csv]: +2509 rows
  BHE[banglahallueval_qa_dataset_1000.csv]: +1000 rows
BHE local total: 4509
BenHalluEval: PAIRBANK 2496 | train rows 4509
bhe train rows: 3496
QA triples: (139618, 5) | BANK size: 104168 | PAIRBANK: 2496


In [7]:
# ===== CELL 7 — CLOZE SYNTHETIC (unchanged) =====
STOP=set("এবং ও কিন্তু বা যে যা এই সেই তার তাদের করা হয় হয়ে হন ছিল ছিলেন একটি একটা এক থেকে সালে জন্য এর কে না নয় করে করেন দিয়ে পরে আগে মধ্যে সাথে হিসেবে".split())
def spans_in(s):
    out=[m.group() for m in DIGIT_RE.finditer(s)]; w=toks(s)
    for n in (1,2,3):
        for i in range(len(w)-n+1):
            g=w[i:i+n]
            if g[0] in STOP or g[-1] in STOP: continue
            if all(len(x)<3 for x in g): continue
            if any(DIGIT_RE.fullmatch(x) for x in g): continue
            sp=" ".join(g)
            if mostly_bengali(sp): out.append(sp)
    return list(dict.fromkeys(out))
def load_wiki(limit):
    files=[]
    for s in ("train/train","valid/valid",""): files+=glob.glob(os.path.join(cfg.wiki_dir,s,"*.txt"))
    random.shuffle(files); out=[]
    for fp in files[:limit]:
        try:
            t=open(fp,encoding="utf-8",errors="ignore").read()
            if len(t)>200 and mostly_bengali(t[:400]): out.append(t[:3000])
        except: pass
    return out
def build_cloze(passages):
    pp=[]
    for p in passages:
        sp=[]
        for s in sent_split(p): sp+=[(s,x) for x in spans_in(s)]
        pp.append(sp)
    rows=[]
    for pi,p in enumerate(passages):
        cand=pp[pi]
        if len(cand)<4: continue
        numc=[(s,x) for s,x in cand if DIGIT_RE.fullmatch(x)]
        sent,span=random.choice(numc) if (numc and random.random()<0.5) else random.choice(cand)
        if span not in sent: continue
        prem=sent.replace(span,"____",1)+" — শূন্যস্থানে কী বসবে? "+p
        rows.append((prem,span,1,"faithful"))
        if DIGIT_RE.fullmatch(span):
            c=bump_digits(span)
            if c: rows.append((prem,c,0,"intrinsic"))
        others=[x for s2,x in cand if x!=span and x not in sent]
        if others: rows.append((prem,random.choice(others),0,"wrong_attr"))
        oj=random.randrange(len(passages))
        if oj!=pi and pp[oj]: rows.append((prem,random.choice(pp[oj])[1],0,"extrinsic"))
    df=pd.DataFrame(rows,columns=["premise","response","label","mode"]); df["src"]="synth"; return df
wiki_passages=load_wiki(cfg.n_wiki_files); print("wiki passages:",len(wiki_passages))
synth_df=build_cloze(wiki_passages) if wiki_passages else pd.DataFrame(columns=["premise","response","label","mode","src"])
print("cloze synthetic:",synth_df.shape)

wiki passages: 7156
cloze synthetic: (24049, 5)


In [8]:
# ===== CELL 8 — ASSEMBLE + MODE-STRATIFIED 50/50 (unchanged from v4.1) =====
if len(nli_df): nli_df=nli_df.assign(mode=nli_df["src"])
parts=[d for d in (bhe_df,qa_df,synth_df,nli_df) if d is not None and len(d)]
train_all=pd.concat([p[["premise","response","label","mode","src"]] for p in parts],ignore_index=True).dropna()
train_all=train_all[train_all["response"].str.len()>0].drop_duplicates(subset=["premise","response"])
train_all=train_all.sample(frac=1,random_state=SEED).reset_index(drop=True)
def cap(df):
    keep=[df[df.src.isin(("bhe","qa"))]]; room=cfg.max_train_rows-len(keep[0])
    for s in ("synth","nli","ixnli"):
        part=df[df.src==s]
        keep.append(part.sample(min(len(part),max(0,room)),random_state=SEED)); room-=len(keep[-1])
    return pd.concat(keep).sample(frac=1,random_state=SEED).reset_index(drop=True)
train_all=cap(train_all)
c1=train_all[train_all.label==1]; c0=train_all[train_all.label==0]
k=min(len(c1),len(c0))
if len(c0)>k:
    tot0=len(c0)
    c0=(c0.groupby("mode",group_keys=False)
          .apply(lambda g:g.sample(max(1,int(round(k*len(g)/tot0))),random_state=SEED)))
    c0=c0.sample(min(len(c0),k),random_state=SEED)
if len(c1)>k: c1=c1.sample(k,random_state=SEED)
train_main=pd.concat([c1,c0]).sample(frac=1,random_state=SEED).reset_index(drop=True)
print("train:",train_main.shape,"| labels:",train_main.label.value_counts().to_dict())
print("class-0 modes:",train_main[train_main.label==0]["mode"].value_counts().to_dict()); tleft()

train: (95222, 5) | labels: {1: 47611, 0: 47611}
class-0 modes: {'wrong_attr': 17467, 'extrinsic': 16522, 'unanswerable': 7633, 'intrinsic': 5989}
[t+2.8m]


In [9]:
# ===== CELL 9 — DATASET · FOCAL · TRAIN/PREDICT (unchanged; fp32 params) =====
class PairDS(Dataset):
    def __init__(self,df,tok,mx,lab=True):
        self.p=df["premise"].astype(str).tolist(); self.h=df["response"].astype(str).tolist()
        self.y=df["label"].tolist() if lab else None; self.t=tok; self.m=mx
    def __len__(self): return len(self.p)
    def __getitem__(self,i):
        e=self.t(self.p[i],self.h[i],truncation=True,max_length=self.m,padding="max_length",return_tensors="pt")
        it={"input_ids":e["input_ids"].squeeze(0),"attention_mask":e["attention_mask"].squeeze(0)}
        if self.y is not None: it["labels"]=torch.tensor(self.y[i],dtype=torch.long)
        return it
class Focal(nn.Module):
    def __init__(self,gamma,alpha):
        super().__init__(); self.g=gamma; self.register_buffer("w",torch.tensor([alpha,1.0]))
    def forward(self,lg,y):
        ce=F.cross_entropy(lg,y,weight=self.w.to(lg.device),reduction="none")
        pt=torch.exp(-ce); return ((1-pt)**self.g*ce).mean()
def resolve_model(name,hf):
    for c in (f"/kaggle/input/{name}",f"/kaggle/input/{name}/{name}"):
        if os.path.exists(os.path.join(c,"config.json")): return c
    return hf
@torch.no_grad()
def predict_proba(model,tok,df,mx,bs):
    model.eval(); out=[]
    for b in DataLoader(PairDS(df,tok,mx,lab=False),batch_size=bs,shuffle=False):
        with torch.amp.autocast("cuda",dtype=torch.float16):
            lg=model(input_ids=b["input_ids"].to(DEVICE),attention_mask=b["attention_mask"].to(DEVICE)).logits.float()
        out.append(torch.softmax(lg,-1)[:,1].cpu().numpy())
    return np.concatenate(out)
def train_backbone(name,hf,tr,val,seed=SEED,quiet=False):
    import copy
    bc=cfg.backbone_cfg.get(name,{"lr":2e-5,"epochs":2}); LR,EPOCHS=bc["lr"],bc["epochs"]
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    path=resolve_model(name,hf); tok=AutoTokenizer.from_pretrained(path)
    model=AutoModelForSequenceClassification.from_pretrained(
        path,num_labels=2,ignore_mismatched_sizes=True).float().to(DEVICE)
    crit=Focal(cfg.focal_gamma,cfg.focal_alpha)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.01)
    ld=DataLoader(PairDS(tr.sample(frac=1,random_state=seed),tok,cfg.max_len),
                  batch_size=cfg.batch_size,shuffle=True,pin_memory=True)
    tot=len(ld)*EPOCHS; sch=get_cosine_schedule_with_warmup(opt,int(tot*cfg.warmup),tot)
    scaler=torch.amp.GradScaler("cuda")
    best=(-1,None,0); curves=[]; tried_retry=False
    ep=0
    while ep<EPOCHS:
        model.train()
        for b in ld:
            opt.zero_grad()
            with torch.amp.autocast("cuda",dtype=torch.float16):
                lg=model(input_ids=b["input_ids"].to(DEVICE),attention_mask=b["attention_mask"].to(DEVICE)).logits
                loss=crit(lg,b["labels"].to(DEVICE))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
        vp=predict_proba(model,tok,val,cfg.max_len,cfg.batch_size*2)
        # COLLAPSE GUARD: single-class val predictions at ep1 => halve LR, reinit, retry once
        spread=float(vp.std())
        if ep==0 and (((vp>=0.5).astype(int).std()==0) or spread<0.01) and not tried_retry:
            print(f"  [COLLAPSE GUARD] {name}: degenerate ep1 (pred-std={spread:.4f}) at lr={LR}."
                  f" Reinitializing at lr={LR/2:.1e} — ELECTRA-large collapses at high LR.")
            LR=LR/2; tried_retry=True
            model=AutoModelForSequenceClassification.from_pretrained(
                path,num_labels=2,ignore_mismatched_sizes=True).float().to(DEVICE)
            opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=0.01)
            sch=get_cosine_schedule_with_warmup(opt,int(tot*cfg.warmup),tot)
            scaler=torch.amp.GradScaler("cuda"); ep=0; curves=[]; continue
        ts=np.quantile(vp,np.linspace(0.05,0.95,40))
        mac=max(f1_score(val["label"],(vp>=t).astype(int),average="macro") for t in ts)
        curves.append({"epoch":ep+1,"lr":LR,"val_best_macro":round(mac,4),"pred_std":round(spread,4)})
        if not quiet: print(f"  {name} lr={LR} ep{ep+1}: val best-MACRO={mac:.4f} (pred-std {spread:.3f})")
        if mac>best[0]: best=(mac,copy.deepcopy({k:v.cpu() for k,v in model.state_dict().items()}),ep+1)
        ep+=1
    model.load_state_dict(best[1]); model.to(DEVICE)
    REPORT.setdefault("train_curves",{})[name]=curves; rep("train_curves",REPORT["train_curves"])
    print(f"  {name}: kept epoch {best[2]} (val MACRO {best[0]:.4f})")
    return model,tok

In [10]:
# ===== CELL 10 — TRAIN banglabert_large ×2 SEEDS (averaged); keep last for retrieval =====
sig_val={}; sig_test={}; keep_for_retr=None
ln,hf=cfg.backbones[0]
pv=[]; pt_=[]
for k in range(cfg.n_seeds_bbl):
    print(f"\n=== {ln} seed {SEED+k} ===")
    m,tk=train_backbone(ln,hf,train_main,sample,seed=SEED+k)
    pv.append(predict_proba(m,tk,sample,cfg.infer_max_len,cfg.batch_size*2))   # infer at 384 (p99 ctx=326)
    pt_.append(predict_proba(m,tk,test,cfg.infer_max_len,cfg.batch_size*2))
    torch.save(m.state_dict(),f"/kaggle/working/{ln}_s{k}.pt")
    if k==cfg.n_seeds_bbl-1 and cfg.use_retrieval:
        keep_for_retr=(m.half().eval(),tk)
    else:
        del m; gc.collect(); torch.cuda.empty_cache()
sig_val[ln]=np.mean(pv,0); sig_test[ln]=np.mean(pt_,0)
from sklearn.metrics import roc_auc_score
seed_corr=float(np.corrcoef(pv[0],pv[1])[0,1]) if len(pv)>1 else 1.0
print(f"seed agreement corr={seed_corr:.3f}")
for reg,mk in (("has",~sample["no_ctx"].values),("no",sample["no_ctx"].values)):
    print(f"{ln} [{reg}] AUC={roc_auc_score(sample.label.values[mk],sig_val[ln][mk]):.3f}")
REPORT.setdefault("encoder",{})["seed_corr"]=round(seed_corr,3); rep("encoder",REPORT["encoder"])
tleft()


=== bbl seed 42 ===


config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

  bbl lr=1e-05 ep1: val best-MACRO=0.7094 (pred-std 0.222)
  bbl lr=1e-05 ep2: val best-MACRO=0.7504 (pred-std 0.232)
  bbl lr=1e-05 ep3: val best-MACRO=0.7238 (pred-std 0.256)
  bbl: kept epoch 2 (val MACRO 0.7504)

=== bbl seed 43 ===


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert_large
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.bias              | UNEXPECTED | 
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the ch

  bbl lr=1e-05 ep1: val best-MACRO=0.6969 (pred-std 0.221)
  bbl lr=1e-05 ep2: val best-MACRO=0.7043 (pred-std 0.219)
  bbl lr=1e-05 ep3: val best-MACRO=0.6975 (pred-std 0.254)
  bbl: kept epoch 2 (val MACRO 0.7043)
seed agreement corr=0.901
bbl [has] AUC=0.922
bbl [no] AUC=0.662
[t+362.5m]
